# 2.1.9 — Computational Complexity and Performance Foundations

**Module:** 02 — NumPy: Advanced Numerical Computing and Array Programming  
**Chapter:** 2.1 — NumPy Architecture and Fundamentals  
**Level:** Expert / Research-Oriented Foundations  
**Prerequisites:** Advanced Python object semantics; basic linear algebra; numerical analysis fundamentals.

> **Purpose.** Develop a precise, testable conceptual and computational model of computational complexity and performance foundations. This notebook separates mathematical intent, Python-level semantics, storage consequences, and execution behavior. It is a standalone section of the Chapter 2.1 architecture sequence.

## 1. Learning Objectives

After studying this notebook, you should be able to:

1. State the central abstractions and identify the level of the system at which each guarantee applies.
2. Explain the relevant mathematical or computational invariants without relying on informal analogies.
3. Construct executable tests that distinguish correct behavior from plausible but invalid assumptions.
4. Identify boundary cases, side effects, dtype concerns, and complexity implications.
5. Translate the topic into a reproducible, documented numerical research workflow.

## 2. Reproducibility and Environment

The examples use the standard `numpy` package and Python standard library. They are intended for recent NumPy 2.x installations; most examples also work on current 1.26 releases, but NumPy 2.x promotion and interface details should be interpreted with the installed version in mind. Execute cells **top to bottom**. There are no external datasets, downloads, or network dependencies.

In [ ]:
import numpy as np
import sys

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 24), 'Use a recent NumPy release.'

## 3. Conceptual Foundations and Formal Semantics

### 1. Asymptotics and constants

Big-O bounds describe scaling, not wall-clock time. Elementwise addition over `N` elements performs O(N) arithmetic and typically O(N) memory traffic. If the output is newly allocated, it requires O(N) storage. A basic reduction is O(N) work but can use a scalar or small auxiliary accumulator, depending on implementation.

### 2. Operational intensity

For a float64 binary addition, an idealized one-pass calculation reads 16 bytes of operands and writes 8 bytes per output: roughly 24 bytes for one floating-point addition. Actual traffic can differ because of caches, allocation, write policies, and broadcasting. Arithmetic intensity is work divided by data transfer, central to a roofline-style interpretation.

### 3. Layout and access costs

Contiguous access usually favors prefetch and spatial locality; large strides can waste cache lines. Yet algorithms must not assume one memory order is always fastest: performance depends on which axis is traversed and which external library is involved.

### 4. Temporary arrays

An expression `a*b + c` may allocate intermediate arrays. Memory footprint, peak allocations, and bandwidth may dominate runtime. `out=` and in-place updates may reduce temporaries when semantics and aliasing permit them.

### 5. Benchmark design

Measure multiple scales, control confounders, and identify what is included in each measurement. Tiny microbenchmarks measure dispatch cost; large arrays may measure memory bandwidth. Any inference about performance is local to the tested environment.

## 4. Architectural Model and Invariants

A useful four-layer checklist for this topic:

| Layer | Question to ask | Representative invariant |
|---|---|---|
| Logical | What operation is mathematically intended? | Axes and result shape correspond to the model |
| Representation | What dtype and storage model represent the inputs? | No unsupported casting or unexpected aliasing |
| Execution | Which category of operation processes the arrays? | Results are stable under a reference implementation |
| Validation | What independent evidence establishes correctness? | Assertions, boundary tests, and numerical tolerances |

**Important:** Shape agreement and code execution are not sufficient evidence of mathematical validity. Conversely, a mathematically valid expression can be operationally impractical or numerically unstable.

## 5. Executable Investigations

Each investigation is independent in purpose; execute in sequence so shared imports are available.

### 5.1 — Analyze theoretical bytes

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
import numpy as np
n = 1_000_000
bytes_per_float = np.dtype(np.float64).itemsize
idealized_bytes = n * 3 * bytes_per_float  # 2 read streams + 1 write stream
print('idealized array traffic (MiB):', idealized_bytes / 2**20)
assert idealized_bytes == 24_000_000

### 5.2 — View versus copy resource use

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
a = np.arange(100_000, dtype=np.float64)
v = a[::2]
c = v.copy()
print('logical bytes:', v.nbytes, 'view strides:', v.strides)
assert np.shares_memory(a, v)
assert not np.shares_memory(a, c)

### 5.3 — Compare representative local access patterns

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
from time import perf_counter
m = np.ones((400, 400), dtype=np.float64)
results = {}
for label, arr in [('contiguous', m), ('transposed_view', m.T)]:
    samples = []
    for _ in range(5):
        t0 = perf_counter(); _ = np.sum(arr, axis=1); samples.append(perf_counter()-t0)
    results[label] = float(np.median(samples))
print(results, 'Note: timings depend on reduction axes and implementation.')

## 6. Edge Cases, Failure Modes, and Misconceptions

- **Logical validity is not physical contiguity.** Shape alone cannot determine whether an operation needs a copy.
- **Value equality is not identity.** Two arrays may compare equal while having independent storage, or share storage while being distinct Python objects.
- **A vectorized expression does not certify correctness.** Broadcasting and axis conventions can silently encode unintended mathematics.
- **A dtype is not a real-number model.** Overflow, finite precision, and representation-dependent transformations require explicit attention.
- **A benchmark does not imply a theorem.** Hardware, build configuration, allocation strategy, and input scale can affect conclusions.
- **Array semantics do not encode domain assumptions.** For finance or statistics, look-ahead restrictions, missing-data rules, weights, and alignment remain external responsibilities.

When evaluating an unexpected result, first inspect `shape`, `dtype`, `strides`, and storage overlap, then isolate the mathematical operation on a hand-checkable example.

## 7. Performance and Complexity Considerations

The performance question should be asked in three parts: **work complexity**, **data movement**, and **peak live memory**. A contiguous numeric array can support efficient native processing, but a particular operation may be limited by memory bandwidth, temporary allocation, scalar dispatch, or native-library parallelism. Use measurements only after verifying correctness, and never assume that a microbenchmark generalizes to different dtypes or problem sizes.

A practical benchmark should (i) isolate the operation, (ii) distinguish input setup from computation, (iii) warm up the relevant execution path, (iv) collect repeated measurements, and (v) record platform and NumPy provenance.

## 8. Analytical Interpretation

**Time complexity, memory complexity, and peak live memory** are distinct quantities. A fast kernel can be slowed by moving data; a low-allocation implementation can still be slow because of poor locality. A correct performance argument states its operational model explicitly.

## 9. Advanced Exercises

These are open-ended expert exercises. Provide an analytical justification, a minimal counterexample where useful, and an executable validation test.

**Exercise 1.** Derive a roofline-style bandwidth lower bound for computing `a*b + c` over float64 arrays.

**Exercise 2.** Explain why benchmark variance grows when arrays exceed LLC capacity.

**Exercise 3.** Compare runtime and peak-memory consequences of fused versus unfused elementwise operations.

## 10. Expected Results and Validation Strategy

1. Execute every example in order; all assertions should pass on an appropriately configured NumPy installation.
2. Compare printed shapes, types, and DTypes with the surrounding mathematical and architectural claims.
3. Where a local benchmark is included, record measured values but **do not** expect a universal numeric timing.
4. Modify input shapes, dtypes, and views to construct at least one failing case; document why the original contract no longer holds.
5. Implement an independent reference for one operation and compare using an error metric appropriate to its numerical domain.

**Acceptance criteria:** the student can explain the behavior without relying on a copied output, design a counterexample to an invalid inference, and describe relevant representation and performance limitations.

## Further Reading and Verification Sources

The following are authoritative public references. API details may vary across NumPy versions; verify against the installed version when reproducing an experiment.

- [NumPy: The N-dimensional array](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
- [NumPy: ndarray reference](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)
- [NumPy: Data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy: Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [NumPy: Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)
- [NumPy: Thread safety](https://numpy.org/doc/stable/reference/thread_safety.html)
- [NumPy: C API](https://numpy.org/doc/stable/reference/c-api/index.html)

**Scope note:** This is Chapter 2.1 (architectural foundations). Detailed DType implementation, advanced indexing, stride tricks, microarchitecture, and native C extension development are treated in Chapters 2.3, 2.5, 2.7, 2.21 and 2.23 respectively.